In [0]:
%run ../init_framework

In [0]:
# # 1. Initialize the CDF Read Stream
# df_bronze_customers = (spark.readStream
#     .format("delta")
#     .option("readChangeFeed", "true") 
#     .option("startingVersion", 1) # Required for the very first execution
#     .table(CUSTOMERS_BRONZE))

In [0]:
# 1. Initialize the CDF Read Stream
df_bronze_customers = (spark.read
    .format("delta")
    .option("readChangeFeed", "true") 
    .option("startingVersion", 1) # Required for the very first execution
    .table(CUSTOMERS_BRONZE))
df_bronze_customers.limit(5).display()

In [0]:
CUSTOMER_RENAME_MAP = {
    "annual_inc": "annual_income",
    "addr_state": "address_state",
    "zip_code": "address_zipcode",
    "country": "address_country",
    "tot_hi_cred_lim": "total_high_credit_limit", 
    "annual_inc_joint": "joint_annual_income"
}

df_renamed_customers = standardize_column_names(df_bronze_customers, CUSTOMER_RENAME_MAP, strict=True)

In [0]:
df_with_metadata = apply_silver_metadata(df_renamed_customers)

In [0]:
df_with_metadata.count()

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
df_distinct.count()

In [0]:
# Drop customers without annual income (NULL or UNKNOWN)
col_ls = ["annual_income"]
df_valid_customers = drop_critical_nulls(df_distinct, col_ls)

In [0]:
df_valid_customers.count()

In [0]:
# Covert emp_length to int type
from pyspark.sql.functions import col, regexp_replace

df_emplength_cleaned = df_valid_customers.withColumn(
    "emp_length", regexp_replace(col("emp_length"), r"\D+", "").cast("int")
)

In [0]:
# Replace null/unknown emp duration with "-1"
df_emplength_notnull = df_emplength_cleaned.fillna(-1, subset=["emp_length"])

In [0]:
from pyspark.sql.functions import col, length, when, trim, upper, lit

df_customers_final = df_emplength_notnull.withColumn(
    "address_state",
    when(
        length(trim(col("address_state"))) == 2, 
        upper(trim(col("address_state")))
    )
    .otherwise(lit("NA")) 
)